# 牛顿环曲率半径计算 (逐差法)

本 Notebook 用于通过读数显微镜测得的牛顿环干涉暗环直径平方数据，利用**逐差法**计算平凸透镜的曲率半径 $R$，并完成高精度的合成不确定度评定与结果修约。

---
### 实验原理与数学公式

#### 1. 牛顿环曲率半径计算公式
平凸透镜与平板玻璃构成的空气薄膜发生等厚干涉，第 $m$ 级暗环直径为 $D_m$。第 $i+M$ 级与第 $i$ 级暗环直径平方差为：
$$\Delta_i = D_{i+M}^2 - D_i^2 = 4 M R \lambda$$
逐差平均值：
$$\Delta_{\mathrm{avg}} = \frac{1}{M} \sum_{i=1}^M \Delta_i$$
曲率半径 $R$ 为：
$$R = \frac{\Delta_{\mathrm{avg}}}{4 M \lambda}$$
* $\lambda$：单色光波长（钠黄光默认取 $\lambda = 589.3\,\mathrm{nm} = 0.0005893\,\mathrm{mm}$）
* $M$：逐差跨度（通常第 11~30 环，分为 11~20 环与 21~30 环，跨度 $M = 10$）

#### 2. 不确定度传递公式
* **A 类不确定度**（针对逐差后的 $\Delta_i$ 项）：
$$s_\Delta = \sqrt{\frac{1}{M-1} \sum_{i=1}^M (\Delta_i - \Delta_{\mathrm{avg}})^2}, \quad u_A(\Delta) = \frac{s_\Delta}{\sqrt{M}}$$
* **B 类不确定度**（由读数显微镜测量误差传递）：
设显微镜坐标不确定度为 $u(x) = \frac{\Delta_{\mathrm{inst}}}{\sqrt{3}}$，则直径 $D = x_R - x_L$ 的平方 $D^2$ 的不确定度满足 $u^2(D^2) = 8 D^2 u^2(x)$。
由逐差平均传播可得：
$$u_B^2(\Delta_{\mathrm{avg}}) = \frac{8 u^2(x)}{M^2} \sum_{i=1}^M (D_i^2 + D_{i+M}^2)$$
* **合成不确定度传递到曲率半径 $R$**：
$$u(\Delta) = \sqrt{u_A^2(\Delta) + u_B^2(\Delta_{\mathrm{avg}})}, \quad u(R) = \frac{u(\Delta)}{4 M \lambda}$$

In [ ]:
import math
from decimal import Decimal
from python.utils import scientific_round

print("牛顿环计算模块加载完成。")

### 1. 实验常数与暗环测量数据输入
> **提示**：`d_squared_vals` 为第 11 到 30 环的暗环直径平方 $D^2\,(\mathrm{mm^2})$（共 20 个数据）。

In [ ]:
# 钠光波长 λ (mm)
WAVELENGTH = Decimal("0.0005893")

# 仪器参数
delta_inst = Decimal("0.005")  # 读数显微镜仪器误差限 (mm)

# 第 11 环 ~ 第 30 环的 D^2 数据 (mm^2)
# 示例数据 (请按实际实验数据替换)：
d_squared_vals = [
    Decimal("12.15"), Decimal("13.20"), Decimal("14.35"), Decimal("15.42"), Decimal("16.55"),
    Decimal("17.68"), Decimal("18.72"), Decimal("19.85"), Decimal("20.90"), Decimal("22.05"),
    Decimal("23.15"), Decimal("24.28"), Decimal("25.40"), Decimal("26.50"), Decimal("27.62"),
    Decimal("28.75"), Decimal("29.80"), Decimal("30.95"), Decimal("32.00"), Decimal("33.15")
]

print(f"光波波长 λ: {WAVELENGTH} mm")
print(f"显微镜误差 Δ_inst: {delta_inst} mm")
print(f"已载入暗环平方数据点数: {len(d_squared_vals)}")

### 2. 逐差计算与曲率半径求解

In [ ]:
M = 10  # 逐差项数
deltas = []
sum_d_pairs = Decimal("0")

print("--- 逐差过程 (Δ_i = D_{i+10}^2 - D_i^2) ---")
for i in range(10):
    d_low_sq = d_squared_vals[i]
    d_high_sq = d_squared_vals[i + 10]
    diff = d_high_sq - d_low_sq
    deltas.append(diff)
    sum_d_pairs += (d_low_sq + d_high_sq)
    print(f"Δ_{i+11:02d} = D_{i+21}^2 - D_{i+11}^2 = {diff:.3f} mm^2")

mean_delta = sum(deltas) / Decimal(str(M))

# 曲率半径 R = Δ_avg / (4 * M * λ)
denominator = Decimal("4") * Decimal(str(M)) * WAVELENGTH
r_val = mean_delta / denominator

# 不确定度计算
# 1. A 类不确定度
variance_delta = sum((x - mean_delta)**2 for x in deltas) / Decimal(str(M - 1))
s_delta = Decimal(str(math.sqrt(float(variance_delta))))
u_a_delta = s_delta / Decimal(str(math.sqrt(M)))

# 2. B 类不确定度传播
u_x = delta_inst / Decimal(str(math.sqrt(3)))
u_b_delta_sq = (Decimal("8") * (u_x**2) * sum_d_pairs) / (Decimal(str(M))**2)
u_b_delta = Decimal(str(math.sqrt(float(u_b_delta_sq))))

# 3. 合成不确定度并在 R 上传递
u_delta_combined = Decimal(str(math.sqrt(float(u_a_delta**2 + u_b_delta**2))))
u_r = u_delta_combined / denominator

# 4. 科学修约
r_final, u_final = scientific_round(r_val, u_r)

print("\n" + "=" * 45)
print("           牛 顿 环 曲 率 半 径 计 算 结 果          ")
print("=" * 45)
print(f"逐差平均差值 Δ_avg  : {mean_delta:.4f} mm^2")
print(f"差值标准差 s(Δ)     : {s_delta:.4f} mm^2")
print(f"A 类不确定度 u_A(Δ) : {u_a_delta:.4f} mm^2")
print(f"B 类不确定度 u_B(Δ) : {u_b_delta:.4f} mm^2")
print("-" * 45)
print(f"曲率半径 R (原始)   : {r_val:.4f} mm ({r_val/1000:.4f} m)")
print(f"合成不确定度 u(R)   : {u_r:.4f} mm")
print("-" * 45)
print(f"★ 最终修约结果      : R = {r_final} ± {u_final} mm")
print("=" * 45)